In [0]:
# =================================================
# PUBLISH: Push Gold Data to Snowflake
# =================================================

catalog = "he_prod_incen_analytics_dbw_01"

# Snowflake Connection Details
sf_url = "jdbc:snowflake://ZQHVRUR-DK53940.snowflakecomputing.com"
sf_user = "UNDERDOG01012000"
sf_password = "Rushikesh@2902" # <--- PASTE YOUR PASSWORD HERE
sf_role = "ACCOUNTADMIN" 
sf_warehouse = "BI_ANALYTICS_WH"
sf_database = "HDFC_ERGO_DW"
sf_schema = "GOLD"

# Build JDBC URL
full_jdbc_url = f"{sf_url}?warehouse={sf_warehouse}&role={sf_role}&db={sf_database}&schema={sf_schema}"

# List of Gold tables to push to Snowflake (Must match your Databricks Gold table names)
gold_tables = [
    "fact_claims",
    "dim_members",
    "dim_geography",
    "dim_providers",
    "dim_facilities"
]

for table_name in gold_tables:
    print(f"🚀 Publishing {table_name} to Snowflake...")
    
    # 1. Read from Databricks Gold
    df_gold = spark.read.table(f"{catalog}.gold.{table_name}")
    
    # 2. Write to Snowflake
    # Using overwrite mode so if you run this again, it replaces the data cleanly
    df_gold.write \
      .format("jdbc") \
      .option("url", full_jdbc_url) \
      .option("dbtable", table_name.upper()) \
      .option("user", sf_user) \
      .option("password", sf_password) \
      .option("driver", "net.snowflake.client.jdbc.SnowflakeDriver") \
      .option("truncate", "true") \ 
      .mode("overwrite") \
      .save()
    
    count = df_gold.count()
    print(f"✅ Successfully pushed {count} rows to Snowflake table: {table_name.upper()}")

print("\n🎉 FULL PUBLISH TO SNOWFLAKE COMPLETE!")